# Práctica 1 - Parte 1: Support Vector Machines

## Paso 0 Preprocesamiento y Pipeline de Datos

Transformamos los datos brutos en matrices numéricas limpias para evitar fugas de información.

### 1 Ingeniería y Limpieza
* **FamilySize** Agrupamos SibSp y Parch en una sola métrica de contexto familiar para simplificar el modelo.
* **Eliminación de ruido** Descartamos columnas como Name o Ticket porque su alta cardinalidad aporta ruido y riesgo de sobreajuste.

### 2 Pipeline de Transformación

Usamos un ColumnTransformer para automatizar el tratamiento según la naturaleza del dato.

| Tipo de Dato | Tratamiento | Justificación |
| :--- | :--- | :--- |
| **Numérico** | SimpleImputer (mediana) | La mediana es más robusta que la media frente a valores extremos. |
| **Categórico** | OneHotEncoder | Convierte texto en valores binarios para que el modelo pueda procesarlos. |

### 3 Partición Estratificada
Dividimos los datos en 80/20 usando stratify=y para mantener la proporción real de supervivientes en ambos conjuntos y asegurar una evaluación objetiva.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split

# Carga
df = pd.read_csv('data/titanic.csv')

# Feature engineering
df_work = df.copy()
df_work['FamilySize'] = df_work['SibSp'] + df_work['Parch'] + 1
df_work.drop(['PassengerId', 'Name', 'Ticket', 'Cabin'], axis=1, inplace=True)

# Split
X = df_work.drop('Survived', axis=1)
y = df_work['Survived']
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Preprocesador para convertir todo a numérico
numeric_features = ['Age', 'Fare', 'SibSp', 'Parch', 'FamilySize']
categorical_features = ['Pclass', 'Sex', 'Embarked']

numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median'))
])
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore'))
])
preprocessor = ColumnTransformer(transformers=[
    ('num', numeric_transformer, numeric_features),
    ('cat', categorical_transformer, categorical_features)
])

# Transformar: X_train y X_test ya quedan numéricos y sin nulos
X_train = preprocessor.fit_transform(X_train_raw)
X_test = preprocessor.transform(X_test_raw)

print(f"Train: {X_train.shape}, Test: {X_test.shape}")
print("Preprocesamiento listo. X_train y X_test ya son numéricos y sin nulos.")

## Verificación de los Datos Finales

Confirmamos que la transformación se ha realizado correctamente y que los conjuntos están listos para el modelado.

### Dimensiones y Estado


| Conjunto | Registros | Columnas | Estado |
| :--- | :--- | :--- | :--- |
| **Entrenamiento** | 712 | 10 | Numérico y sin nulos |
| **Test** | 179 | 10 | Numérico y sin nulos |

### Validación de la Estructura
Comprobamos las dimensiones para asegurar que el preprocesador ha mantenido la consistencia en el número de características entre ambos sets. Esto evita errores de compatibilidad durante el entrenamiento y la fase de predicción.

> **Conclusión** El dataset está optimizado y preparado para alimentar al algoritmo.

---
### 1.1 SVM con Diferentes Kernels
Apartado 1.1 del enunciado. Se entrenan 4 SVMs con kernels distintos y se compara su rendimiento con hiperparámetros por defecto.


## 1.1 Evaluación de Kernels en SVM

Entrenamos cuatro configuraciones de Máquinas de Vector de Soporte para comparar cómo afectan los diferentes kernels a la frontera de decisión.

### Escalado Obligatorio

Las SVM basan su cálculo en distancias por lo que aplicamos un StandardScaler dentro de un Pipeline. Esto garantiza que todas las variables tengan el mismo peso y evita que el modelo ignore características con rangos numéricos menores.

### Configuración de Kernels
Probamos cuatro aproximaciones distintas para proyectar los datos y encontrar el hiperplano óptimo.
* **Linear** Busca una separación lineal directa entre clases.
* **Poly (grado 3)** Crea una frontera polinómica para capturar relaciones no lineales.
* **RBF** Utiliza funciones de base radial para manejar distribuciones circulares o complejas.
* **Sigmoid** Aplica una transformación similar a las funciones de activación de redes neuronales.


Usamos Validación Cruzada con 5 particiones para asegurar la estabilidad del modelo. Monitorizamos el número de Vectores de Soporte (SV) porque un porcentaje demasiado alto respecto al total de entrenamiento suele ser un indicador de sobreajuste (*overfitting*).

> El objetivo es encontrar el kernel que maximice las métricas de clasificación manteniendo un número razonable de vectores de soporte.

In [ ]:
from sklearn.svm import SVC
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

# IMPORTANTE: SVM requiere escalado obligatorio
kernels = {
    'Linear':     SVC(kernel='linear',  random_state=42),
    'Poly (d=3)': SVC(kernel='poly',    degree=3, random_state=42),
    'RBF':        SVC(kernel='rbf',     random_state=42),
    'Sigmoid':    SVC(kernel='sigmoid', random_state=42)
}

for name, svc in kernels.items():
    pipe = Pipeline([('scaler', StandardScaler()), ('svm', svc)])
    pipe.fit(X_train, y_train)
    scores = cross_val_score(pipe, X_train, y_train, cv=5, scoring='accuracy')
    n_sv = pipe.named_steps['svm'].n_support_
    print(f"{name}: CV={scores.mean():.4f} (+/-{scores.std():.4f}), "
          f"SV={n_sv.sum()} ({n_sv.sum()/len(X_train)*100:.1f}%)")

Tabla que pide la práctica: Kernel | CV Accuracy | Test Accuracy | Precision | Recall | F1-Score | N° SV | % Training.

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

rows = []
for name, svc in kernels.items():
    pipe = Pipeline([('scaler', StandardScaler()), ('svm', svc)])
    pipe.fit(X_train, y_train)
    scores = cross_val_score(pipe, X_train, y_train, cv=5, scoring='accuracy')
    y_pred = pipe.predict(X_test)
    n_sv = pipe.named_steps['svm'].n_support_
    rows.append({
        'Kernel': name,
        'CV Accuracy': f"{scores.mean():.4f}",
        'Test Accuracy': f"{accuracy_score(y_test, y_pred):.4f}",
        'Precision': f"{precision_score(y_test, y_pred):.4f}",
        'Recall': f"{recall_score(y_test, y_pred):.4f}",
        'F1-Score': f"{f1_score(y_test, y_pred):.4f}",
        'N° SV': n_sv.sum(),
        '% Training': f"{n_sv.sum()/len(X_train)*100:.1f}%"
    })
pd.DataFrame(rows)

## Análisis de Resultados de Kernels

Comparamos el desempeño de los modelos para identificar cuál captura mejor los patrones de supervivencia del Titanic.

### El Ganador Es RBF
El kernel **RBF** obtiene la mejor precisión tanto en validación cruzada (83.85%) como en test (80.45%). Es el modelo más equilibrado con el F1-Score más alto (0.7059). Su flexibilidad matemática le permite adaptarse a la frontera de decisión compleja de este problema sin caer en un sobreajuste excesivo.

### El Comportamiento del Poly (d=3)
Aunque empata en precisión total con RBF muestra un sesgo interesante
* **Alta Precisión (86.96%)** Cuando el modelo predice que alguien sobrevive casi siempre acierta.
* **Bajo Recall (57.97%)** Se le escapan muchos supervivientes reales (falsos negativos).
Es un modelo conservador que solo se arriesga a predecir "sobrevive" cuando la evidencia es muy clara.

### Rendimiento y Complejidad
* **Fracaso del Sigmoid** Es el kernel menos eficiente con una caída drástica en todas las métricas porque su transformación no encaja con la estructura lógica de los datos del Titanic.
* **Vectores de Soporte** El uso del 46% de los datos como soporte confirma que las clases están muy solapadas. Esto obliga al modelo a emplear casi la mitad del dataset para definir una frontera de decisión fiable.

> **Conclusión** El kernel **RBF** es la opción más robusta porque ofrece la mejor capacidad de generalización y el mejor equilibrio entre precisión y recuperación.

---
## 1.2 Optimización del Kernel RBF

Afinamos el modelo mediante GridSearchCV para encontrar el equilibrio óptimo entre sesgo y varianza explorando el espacio de hiperparámetros.

### Estrategia de Búsqueda
Exploramos diferentes combinaciones de C y gamma para maximizar la capacidad predictiva del kernel radial. 



| Parámetro | Función | Efecto |
| :--- | :--- | :--- |
| **C** | Regularización | Controla la penalización por errores. Un valor alto busca clasificar todo bien pero arriesga sobreajuste. |
| **gamma** | Influencia | Define el radio de acción de los puntos. Un valor alto genera fronteras más complejas y ajustadas. |

### Análisis de Complejidad y Rendimiento
Evaluamos las mejores combinaciones considerando el acierto y la eficiencia del modelo.

* **Vectores de Soporte** Un porcentaje alto indica que el modelo depende de muchos puntos para definir la frontera lo que sugiere que los datos están muy solapados.
* **Eficiencia Temporal** Monitorizamos el tiempo de entrenamiento para asegurar que el modelo sea escalable y práctico.

> **Estado Final** El DataFrame resultante permite identificar la configuración que maximiza la precisión en test manteniendo una estructura de vectores de soporte equilibrada.

In [ ]:
from sklearn.model_selection import GridSearchCV

pipe_rbf = Pipeline([
    ('scaler', StandardScaler()),
    ('svm', SVC(kernel='rbf', random_state=42))
])

param_grid = {
    'svm__C':     [0.1, 1, 10, 100],
    'svm__gamma': [0.001, 0.01, 0.1, 1, 'scale', 'auto']
}

grid_rbf = GridSearchCV(pipe_rbf, param_grid, cv=5, scoring='accuracy', n_jobs=-1)
grid_rbf.fit(X_train, y_train)

print(f"Mejores parámetros: {grid_rbf.best_params_}")
print(f"Mejor CV score:     {grid_rbf.best_score_:.4f}")
print(f"Test score:         {grid_rbf.best_estimator_.score(X_test, y_test):.4f}")

Tabla con al menos 10 combinaciones representativas del GridSearch RBF.

In [ ]:
from time import time

results_rbf = pd.DataFrame(grid_rbf.cv_results_)
results_rbf = results_rbf.sort_values('mean_test_score', ascending=False)

rows = []
for _, r in results_rbf.head(15).iterrows():
    p = Pipeline([('scaler', StandardScaler()),
                  ('svm', SVC(kernel='rbf', C=r['param_svm__C'],
                              gamma=r['param_svm__gamma'], random_state=42))])
    t0 = time()
    p.fit(X_train, y_train)
    t1 = time()
    n_sv = p.named_steps['svm'].n_support_.sum()
    rows.append({
        'C': r['param_svm__C'], 'gamma': r['param_svm__gamma'],
        'CV Accuracy': f"{r['mean_test_score']:.4f}",
        'Test Accuracy': f"{p.score(X_test, y_test):.4f}",
        'N° SV': n_sv,
        '% Training': f"{n_sv/len(X_train)*100:.1f}%",
        'Tiempo (s)': f"{t1-t0:.3f}"
    })
pd.DataFrame(rows)

## Análisis de Optimización RBF

Tras evaluar las combinaciones del grid search identificamos la configuración que maximiza la precisión sin comprometer la estabilidad del modelo.

### Selección de Hiperparámetros
Los resultados muestran que **C=1** y **gamma=0.1** son los valores óptimos. El hecho de que `scale` y `auto` obtengan métricas idénticas confirma que el preprocesamiento de las variables ha sido efectivo y que la escala de los datos es uniforme.

### Comportamiento de la Frontera y Complejidad
Analizamos cómo cambia el modelo según los parámetros para entender el riesgo de sobreajuste
* **Efecto de C** Al subir a C=100 el acierto en test cae drásticamente a 0.7709. El modelo se vuelve demasiado estricto con los errores de entrenamiento y pierde flexibilidad frente a datos nuevos.
* **Efecto de gamma** Con gamma=1 el número de vectores de soporte sube al 61.9% del dataset. Esto indica una frontera de decisión demasiado sinuosa que intenta rodear puntos individuales en lugar de capturar la tendencia general.

### Rendimiento Final

El modelo seleccionado ofrece un Test Accuracy de 0.8045. Aunque existen combinaciones con un acierto puntual en test algo mayor (como C=10 y gamma=0.01) mantenemos la ganadora de la validación cruzada porque es la más consistente estadísticamente.

> **Conclusión** La configuración elegida garantiza un modelo equilibrado que utiliza el 45.8% de los datos como soporte para definir una frontera robusta y eficiente.

---
### 1.3 Optimización del Kernel Polinomial

A continuación, optimizamos un **Kernel Polinomial**, que nos permite encontrar fronteras de decisión no lineales al combinar las características existentes.

Usamos de nuevo un `Pipeline` con `GridSearchCV` para evaluar 27 combinaciones distintas de hiperparámetros (con 5 iteraciones de validación cruzada cada una):
- **degree** (2, 3 y 4): Controla la complejidad de la frontera (a mayor grado, más riesgo de sobreajuste y mayor coste computacional).
- **C** (0.1, 1 y 10): Regula la penalización de los errores.
- **gamma** ('scale', 'auto', 0.1): Determina la escala del kernel.

El proceso está paralelizado (`n_jobs=-1`) para aprovechar todos los núcleos del procesador y reducir el tiempo de búsqueda de los mejores parámetros.


In [ ]:
pipe_poly = Pipeline([
    ('scaler', StandardScaler()),
    ('svm', SVC(kernel='poly', random_state=42))
])

param_grid = {
    'svm__degree': [2, 3, 4],
    'svm__C':      [0.1, 1, 10],
    'svm__gamma':  ['scale', 'auto', 0.1]
}

grid_poly = GridSearchCV(pipe_poly, param_grid, cv=5, scoring='accuracy', n_jobs=-1)
grid_poly.fit(X_train, y_train)

Tabla de resultados del GridSearch polinomial.

In [ ]:
results_poly = pd.DataFrame(grid_poly.cv_results_)
results_poly = results_poly.sort_values('mean_test_score', ascending=False)

rows = []
for _, r in results_poly.iterrows():
    p = Pipeline([('scaler', StandardScaler()),
                  ('svm', SVC(kernel='poly', degree=int(r['param_svm__degree']),
                              C=r['param_svm__C'], gamma=r['param_svm__gamma'],
                              random_state=42))])
    p.fit(X_train, y_train)
    n_sv = p.named_steps['svm'].n_support_.sum()
    rows.append({
        'degree': int(r['param_svm__degree']),
        'C': r['param_svm__C'],
        'gamma': r['param_svm__gamma'],
        'CV Accuracy': f"{r['mean_test_score']:.4f}",
        'Test Accuracy': f"{p.score(X_test, y_test):.4f}",
        'N° SV': n_sv
    })
pd.DataFrame(rows)

### Análisis de Resultados: Optimización del Kernel Polinomial

El proceso de validación cruzada con **GridSearchCV** ha dejado conclusiones muy claras sobre los hiperparámetros en este kernel.

* **El grado 3 es el punto óptimo**.
Este valor ha demostrado ser el factor más determinante para nuestro conjunto de datos con una precisión en validación cruzada de **0.8287** y de **0.8045** en el conjunto de prueba. El **grado 2** sufre un ajuste insuficiente mientras que el **grado 4** empieza a perder rendimiento por sobreajuste. En el contexto del Titanic las relaciones de tercer orden son más que suficientes para trazar una frontera no lineal efectiva.

* **Impacto del parámetro de penalización C**.
El valor **1.0** ofrece el mejor equilibrio posible para el modelo. Si bajamos la exigencia a **0.1** la precisión cae de forma drástica hasta el rango entre **0.65** y **0.77**. En ese escenario el número de **vectores de soporte** se dispara hasta los **538** porque el margen es tan permisivo que casi cualquier punto termina influyendo en la frontera y se pierde la capacidad de discriminación. Por otro lado subir a un valor de **10** no aporta mejoras y solo crea una frontera demasiado estricta que generaliza peor.

* **La escala de gamma no influye**.
No se ha observado ninguna alteración en los resultados al cambiar entre las opciones de **auto** o **scale** o el valor fijo de **0.1**. Las métricas de precisión y el número de **vectores de soporte** se mantienen fijos en **330** para las mejores combinaciones encontradas.

**Conclusión final**:
La configuración de **grado 3** y **C de 1.0** apoyada en **330 vectores de soporte** es la más potente para este kernel. A pesar de su buen rendimiento este modelo no logra superar los resultados del **kernel RBF** que ha demostrado ser ligeramente más preciso al gestionar las interacciones entre las variables de este dataset.

---
### 1.4 Comparación de Implementaciones (Kernel Lineal)

En esta sección vamos a poner a prueba tres versiones diferentes de un **clasificador lineal** para ver cuál se comporta mejor con nuestro dataset.

Probamos el **SVC convencional** con kernel lineal junto a la implementación optimizada de **LinearSVC** y el **SGDClassifier** configurado con pérdida **hinge**. Aunque en teoría los tres deberían dar resultados parecidos sus algoritmos internos de optimización son distintos y eso suele influir tanto en la velocidad como en la precisión final.

Para que la comparativa sea justa usamos un **Pipeline** que incluye el escalado de datos con **StandardScaler** de forma automática. El objetivo es evaluar la **precisión en validación cruzada** y en el conjunto de **test** fijándonos también en los **tiempos de ejecución** para elegir la opción más eficiente.

In [ ]:
from sklearn.svm import SVC, LinearSVC
from sklearn.linear_model import SGDClassifier
from time import time

models = {
    'SVC(linear)':   Pipeline([('scaler', StandardScaler()),
                                ('svm', SVC(kernel='linear', random_state=42))]),
    'LinearSVC':     Pipeline([('scaler', StandardScaler()),
                                ('svm', LinearSVC(random_state=42, max_iter=10000))]),
    'SGDClassifier': Pipeline([('scaler', StandardScaler()),
                                ('svm', SGDClassifier(loss='hinge', random_state=42))])
}

for name, model in models.items():
    t0 = time()
    model.fit(X_train, y_train)
    t1 = time()
    cv = cross_val_score(model, X_train, y_train, cv=5)
    print(f"{name}: CV={cv.mean():.4f}, Test={model.score(X_test, y_test):.4f}, "
          f"Tiempo={t1-t0:.3f}s")

Tabla como pide el enunciado. N° SV solo disponible en SVC.

In [ ]:
rows = []
for name, model in models.items():
    t0 = time()
    model.fit(X_train, y_train)
    t1 = time()
    cv = cross_val_score(model, X_train, y_train, cv=5)
    n_sv = 'N/A'
    if hasattr(model.named_steps['svm'], 'n_support_'):
        n_sv = model.named_steps['svm'].n_support_.sum()
    rows.append({
        'Implementación': name,
        'CV Accuracy': f"{cv.mean():.4f}",
        'Test Accuracy': f"{model.score(X_test, y_test):.4f}",
        'N° SV': n_sv,
        'Tiempo (s)': f"{t1-t0:.3f}"
    })
pd.DataFrame(rows)

### Análisis de Resultados de la Comparativa Lineal

Los datos muestran diferencias muy claras entre las tres formas de entrenar un modelo lineal para este problema.

**LinearSVC** es la mejor opción de largo. Ha sacado la precisión más alta con un **0.7936** en validación y un **0.7933** en el test. Además de ser el más equilibrado es mucho más rápido que el SVC normal al estar optimizado específicamente para casos lineales.

El **SVC con kernel lineal** cumple bien pero se queda un poco por detrás. Ha conseguido un **0.7765** en el test apoyándose en **332 vectores de soporte**. Es una alternativa sólida aunque tarda algo más en procesar los datos y no ajusta tan bien como el primero.

El **SGDClassifier** es increíblemente rápido pero no merece la pena. Su precisión en el test baja hasta el **0.7095** y eso indica que es demasiado inestable para este dataset concreto. La brecha tan grande entre la validación y el test sugiere que el algoritmo no ha llegado a converger correctamente.

**Conclusión**:
El ganador indiscutible es **LinearSVC** por su gran capacidad de generalización y su rapidez. El modelo lineal estándar es aceptable pero el SGD no es una buena elección si buscamos resultados precisos.

---
### 1.5 Análisis de Vectores de Soporte

En esta parte vamos a comprobar cuántos puntos del dataset está usando el modelo de verdad para marcar el límite entre las clases. 

Aplicamos los mejores valores de **C** y **gamma** que encontramos antes para el kernel **RBF** y calculamos el total de **vectores de soporte**. Saber qué porcentaje del entrenamiento representan es fundamental porque nos indica si el modelo es demasiado complejo o si ha logrado un buen equilibrio.

In [ ]:
# Usamos los mejores C y gamma del apartado 1.2
best_C = grid_rbf.best_params_['svm__C']
best_gamma = grid_rbf.best_params_['svm__gamma']

# El enunciado usa X_train_scaled directamente
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)

model_svc = SVC(kernel='rbf', C=best_C, gamma=best_gamma, random_state=42)
model_svc.fit(X_train_scaled, y_train)

print(f"Total de vectores de soporte: {model_svc.n_support_.sum()}")
print(f"Por clase:                    {model_svc.n_support_}")
print(f"Porcentaje del training set:  {model_svc.n_support_.sum()/len(X_train)*100:.2f}%")

Tabla comparativa de SV con distintas configuraciones.

In [ ]:
configs = {
    'Linear (C=1)':         SVC(kernel='linear', C=1, random_state=42),
    'RBF (C=1, γ=scale)':   SVC(kernel='rbf', C=1, gamma='scale', random_state=42),
    'RBF (C=10, γ=0.1)':    SVC(kernel='rbf', C=10, gamma=0.1, random_state=42),
    'RBF (C=100, γ=1)':     SVC(kernel='rbf', C=100, gamma=1, random_state=42),
    'Poly (d=3, C=1)':      SVC(kernel='poly', degree=3, C=1, random_state=42)
}

rows = []
for name, svc in configs.items():
    svc.fit(X_train_scaled, y_train)
    n_sv = svc.n_support_.sum()
    rows.append({
        'Configuración': name,
        'N° SV': n_sv,
        '% Training Set': f"{n_sv/len(X_train)*100:.2f}%"
    })
pd.DataFrame(rows)

### Análisis de Resultados: Vectores de Soporte

Los datos de la tabla permiten ver claramente cómo afecta cada configuración a la complejidad del modelo.

El modelo **Linear (C=1)** y el **RBF con C=1 y γ=scale** muestran resultados muy similares junto al **Polinomial**. Los tres se mantienen cerca del **46%** de vectores de soporte. Esto nos indica que el modelo necesita una cantidad de puntos considerable para separar las clases pero sin llegar a saturar la memoria del algoritmo.

La mejor opción en cuanto a limpieza es el **RBF con C=10 y γ=0.1**. Solo necesita un **42.84%** de los datos para definir la frontera y eso lo convierte en el modelo más eficiente de la comparativa. Logra un margen muy sólido usando menos puntos de apoyo que el resto.

El caso del **RBF con C=100 y γ=1** es el más problemático porque el número de vectores se dispara hasta el **57.72%**. Superar la mitad del dataset para marcar el límite es una prueba de que el modelo está memorizando el ruido. Esta configuración es la que tiene más riesgo de sufrir sobreajuste y no funcionará bien con datos nuevos.

**Conclusión**:
Lo normal para este problema es moverse en torno al **45%** de vectores de soporte. Debemos evitar los parámetros que suban este porcentaje por encima del **50%** para garantizar que el clasificador sea capaz de generalizar correctamente.

---
### 1.6 Evaluación Final (Parte 1)

En este apartado vamos a analizar a fondo el rendimiento del mejor modelo que hemos encontrado. Sacamos el **classification report** y la **matriz de confusión** para ver exactamente cómo está clasificando cada categoría sobre el conjunto de test.

También generamos una tabla comparativa con la versión ganadora de cada kernel. Así podremos ver de un vistazo cuál es el modelo que mejor generaliza y cuántos **vectores de soporte** necesita cada uno para tomar sus decisiones finales.

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

best_svm = grid_rbf.best_estimator_
best_svm.fit(X_train, y_train)
y_pred = best_svm.predict(X_test)

print(classification_report(y_test, y_pred))

cm = confusion_matrix(y_test, y_pred)
ConfusionMatrixDisplay(confusion_matrix=cm).plot()
plt.savefig('images/confusion_matrix_svm.png', dpi=300, bbox_inches='tight')
plt.show()

Tabla resumen final de la Parte 1: mejor modelo de cada kernel.

In [ ]:
# Mejor Linear
p_lin = Pipeline([('scaler', StandardScaler()), ('svm', SVC(kernel='linear', random_state=42))])
p_lin.fit(X_train, y_train)
cv_lin = cross_val_score(p_lin, X_train, y_train, cv=5)
yp_lin = p_lin.predict(X_test)

# Mejor Poly (del GridSearch)
best_poly = grid_poly.best_estimator_
best_poly.fit(X_train, y_train)
cv_poly = cross_val_score(best_poly, X_train, y_train, cv=5)
yp_poly = best_poly.predict(X_test)

# Mejor RBF (del GridSearch)
best_rbf = grid_rbf.best_estimator_
best_rbf.fit(X_train, y_train)
cv_rbf = cross_val_score(best_rbf, X_train, y_train, cv=5)
yp_rbf = best_rbf.predict(X_test)

# Sigmoid
p_sig = Pipeline([('scaler', StandardScaler()), ('svm', SVC(kernel='sigmoid', random_state=42))])
p_sig.fit(X_train, y_train)
cv_sig = cross_val_score(p_sig, X_train, y_train, cv=5)
yp_sig = p_sig.predict(X_test)

rows = []
for name, pipe, cv, yp in [('SVM Linear (mejor C)', p_lin, cv_lin, yp_lin),
                            ('SVM Poly (mejor config)', best_poly, cv_poly, yp_poly),
                            ('SVM RBF (mejor config)', best_rbf, cv_rbf, yp_rbf),
                            ('SVM Sigmoid (mejor config)', p_sig, cv_sig, yp_sig)]:
    n_sv = pipe.named_steps['svm'].n_support_.sum()
    rows.append({
        'Modelo': name,
        'CV Accuracy': f"{cv.mean():.4f}",
        'Test Accuracy': f"{accuracy_score(y_test, yp):.4f}",
        'F1-Score': f"{f1_score(y_test, yp):.4f}",
        'N° SV': n_sv,
        '% Training': f"{n_sv/len(X_train)*100:.1f}%"
    })
pd.DataFrame(rows)

### Análisis de Resultados: Evaluación Final

El mejor modelo alcanza un **80% de precisión** en el conjunto de prueba y eso representa un rendimiento muy sólido para este dataset. 

Al observar la matriz de confusión y las tablas vemos que el clasificador es muy fiable para predecir quién no sobrevivió. Ha logrado un **recall de 0.93** en esa clase porque solo ha tenido 8 fallos reales. Sin embargo tiene más dificultades con los supervivientes ya que se le escapan 27 personas que el modelo marca como fallecidas. Esto nos deja una sensibilidad del **0.61** para la clase 1 lo que indica que el modelo es algo conservador a la hora de asignar supervivientes.

En la tabla comparativa el **SVM RBF** destaca como el ganador absoluto. Ha conseguido la mejor puntuación en validación cruzada con un **0.8385** y el **F1-Score** más alto con un **0.7059**. Aunque el modelo **Polinomial** empata en precisión final sobre el test el RBF demuestra ser un poco más equilibrado en sus métricas generales.

El kernel **Lineal** cumple de forma aceptable pero se queda claramente por debajo de las opciones no lineales en todas las métricas de acierto. Por último el **Sigmoid** es la peor elección con diferencia porque su precisión en test cae hasta el **0.6704** y sus resultados son demasiado pobres para considerarlo una opción válida en este problema.

**Conclusión final**:
El kernel **RBF** con su configuración optimizada es la mejor herramienta para este problema. Logra captar mejor la complejidad del Titanic que el resto de modelos y mantiene un porcentaje de vectores de soporte del **45.8%** que garantiza una buena capacidad de generalización.

---
## Parte 2: Árboles de Decisión

### 2.1 Árbol sin Restricciones (Baseline)

En esta sección creamos un **modelo base** entrenando un árbol de decisión sin aplicar ninguna limitación en su crecimiento. El objetivo principal es observar de forma clara el fenómeno del **sobreajuste** o **overfitting**.

Dejamos que el algoritmo genere ramas hasta que todas las hojas sean puras o contengan muy pocos ejemplos. Después calculamos la **precisión** tanto en el conjunto de entrenamiento como en el de prueba además de consultar la **profundidad final** y el **número de hojas** que ha necesitado el modelo para memorizar los datos. Esta comparativa nos servirá para entender por qué es tan importante controlar la complejidad de los árboles en el futuro.

In [ ]:
from sklearn.tree import DecisionTreeClassifier

tree_full = DecisionTreeClassifier(random_state=42)
tree_full.fit(X_train, y_train)

print(f"Train accuracy: {tree_full.score(X_train, y_train):.4f}")
print(f"Test  accuracy: {tree_full.score(X_test,  y_test):.4f}")
print(f"Profundidad:    {tree_full.get_depth()}")
print(f"N° hojas:       {tree_full.get_n_leaves()}")

Tabla CON: Configuración | Train Accuracy | Test Accuracy | Profundidad | N° Hojas.

In [ ]:
rows = [{
    'Configuración': 'Sin restricciones',
    'Train Accuracy': f"{tree_full.score(X_train, y_train):.4f}",
    'Test Accuracy': f"{tree_full.score(X_test, y_test):.4f}",
    'Profundidad': tree_full.get_depth(),
    'N° Hojas': tree_full.get_n_leaves()
}]
pd.DataFrame(rows)

### Análisis de Resultados: Árbol sin Restricciones

Los números reflejan de forma muy clara lo que ocurre cuando dejamos que un árbol crezca sin ningún tipo de control.

Existe una brecha enorme entre la precisión de entrenamiento de **0.9831** y la de test que se queda en **0.8045**. Esta diferencia es el ejemplo perfecto de **sobreajuste** o **overfitting** porque el modelo ha memorizado prácticamente todos los detalles de los datos de entrenamiento pero no es capaz de aplicar ese conocimiento con la misma eficacia en datos nuevos.

La complejidad del modelo es excesiva con una **profundidad de 20 niveles** y un total de **159 hojas**. Para un dataset de este tamaño esos valores indican que el árbol ha creado reglas demasiado específicas para casos particulares en lugar de buscar patrones generales. Aunque la precisión en test no es mala el modelo es demasiado inestable y frágil.

**Conclusión**:
Este árbol es demasiado complejo para ser fiable en un entorno real. Los resultados demuestran que es imprescindible limitar el crecimiento mediante parámetros de poda para conseguir un modelo más sencillo que mantenga o incluso mejore el acierto en el conjunto de prueba.

---
### 2.2 Efecto de la Profundidad Máxima

En este apartado vamos a analizar cómo cambia el rendimiento del árbol al limitar su crecimiento con el parámetro **max_depth**.

Probamos varios niveles de profundidad desde un árbol muy simple con solo un nivel hasta uno sin restricciones para observar la evolución de las métricas. El objetivo es identificar en qué punto el modelo deja de aprender patrones útiles y empieza a memorizar el ruido del entrenamiento.

Calculamos la precisión en **train** y en **test** además de usar **validación cruzada** para asegurar que los resultados sean fiables. También controlamos el **número de hojas** final porque es un indicador directo de la complejidad estructural que alcanza el clasificador.

In [ ]:
depths = [1, 2, 3, 5, 7, 10, 15, None]

for depth in depths:
    tree = DecisionTreeClassifier(max_depth=depth, random_state=42)
    cv   = cross_val_score(tree, X_train, y_train, cv=5)
    tree.fit(X_train, y_train)
    print(f"max_depth={str(depth):>4}: train={tree.score(X_train, y_train):.4f}, "
          f"cv={cv.mean():.4f}, test={tree.score(X_test, y_test):.4f}, "
          f"hojas={tree.get_n_leaves()}")

Tabla con: max_depth | Train Acc | CV Acc | Test Acc | Profundidad real | N° Hojas.

In [ ]:
rows = []
for depth in depths:
    tree = DecisionTreeClassifier(max_depth=depth, random_state=42)
    cv = cross_val_score(tree, X_train, y_train, cv=5)
    tree.fit(X_train, y_train)
    rows.append({
        'max_depth': str(depth),
        'Train Acc': f"{tree.score(X_train, y_train):.4f}",
        'CV Acc': f"{cv.mean():.4f}",
        'Test Acc': f"{tree.score(X_test, y_test):.4f}",
        'Profundidad real': tree.get_depth(),
        'N° Hojas': tree.get_n_leaves()
    })
pd.DataFrame(rows)

### Análisis de Resultados: Efecto de la Profundidad Máxima

La tabla muestra una evolución muy clara del modelo desde la simplicidad absoluta hasta el sobreajuste total.

* **El punto óptimo está en max_depth=3**
Este valor es claramente el ganador de la comparativa. Con esta profundidad el modelo logra su mejor rendimiento en validación cruzada alcanzando un **0.8091** y mantiene un sólido **0.8045** en el conjunto de prueba. Es el nivel donde el árbol es lo suficientemente complejo para captar los patrones importantes pero sigue siendo sencillo con solo **8 hojas**.

* **Bajo ajuste en profundidades pequeñas**
Con valores de 1 o 2 el árbol es demasiado simple para el problema del Titanic. Aunque la diferencia entre train y test es pequeña las precisiones son más bajas porque el modelo no tiene capacidad suficiente para entender las relaciones entre las variables.

* **Sobreajuste a partir de profundidad 7**
A medida que permitimos que el árbol crezca más allá de los 7 niveles empezamos a ver un **overfitting** de manual. La precisión en el entrenamiento sube sin parar hasta rozar el **0.9831** mientras que la validación cruzada cae drásticamente hasta el **0.7417**. Esto significa que el modelo está memorizando el ruido del entrenamiento y pierde su capacidad de acierto con datos nuevos.

**Conclusión del experimento**:
Limitar la profundidad a **3 niveles** es la mejor estrategia para este modelo. Nos da un clasificador muy fácil de interpretar que además es el que mejor generaliza según los datos de validación. Usar profundidades mayores solo genera una estructura innecesaria de hasta **159 hojas** que no aporta ninguna ventaja real en el test.

---
### 2.3 Criterios de División

En este apartado vamos a probar diferentes formas de medir la impureza de los nodos para ver cuál separa mejor los datos del Titanic. 

Comparamos los criterios de **Gini**, **Entropía** y **Log Loss** para comprobar si la métrica de división influye de forma significativa en la estructura del árbol. El objetivo es evaluar la precisión en **validación cruzada** y en el conjunto de **test** además de observar si alguno de estos criterios genera un modelo más sencillo con menos número de hojas.

In [ ]:
criterios = ['gini', 'entropy', 'log_loss']

for criterio in criterios:
    tree = DecisionTreeClassifier(criterion=criterio, random_state=42)
    cv   = cross_val_score(tree, X_train, y_train, cv=5)
    tree.fit(X_train, y_train)
    print(f"{criterio}: CV={cv.mean():.4f} (+/-{cv.std():.4f}), "
          f"Test={tree.score(X_test, y_test):.4f}, hojas={tree.get_n_leaves()}")

Tabla con: Criterio | CV Accuracy | Test Accuracy | N° Hojas.

In [ ]:
rows = []
for criterio in criterios:
    tree = DecisionTreeClassifier(criterion=criterio, random_state=42)
    cv = cross_val_score(tree, X_train, y_train, cv=5)
    tree.fit(X_train, y_train)
    rows.append({
        'Criterio': criterio,
        'CV Accuracy': f"{cv.mean():.4f}",
        'Test Accuracy': f"{tree.score(X_test, y_test):.4f}",
        'N° Hojas': tree.get_n_leaves()
    })
pd.DataFrame(rows)

### Análisis de Resultados: Criterios de División

Las métricas demuestran que el criterio de división no altera radicalmente el rendimiento del modelo en este dataset.

El criterio **Gini** saca una ventaja mínima con un **0.7417** en validación cruzada y el mejor acierto en test de la comparativa alcanzando el **0.8045**. Genera un árbol con **159 hojas** y eso indica una estructura ligeramente más ramificada que sus competidores para intentar captar cada detalle.

Por su parte la **Entropía** y el **Log Loss** han devuelto resultados idénticos entre sí con un **0.7989** en el test. Aunque son un poco menos precisos que Gini consiguen modelos un poco más compactos de **155 hojas** lo que sugiere que estas métricas tienden a agrupar los datos de forma algo más eficiente en este caso concreto.

**Conclusión del apartado**:
La diferencia entre las tres opciones es prácticamente insignificante para el problema del Titanic. Podríamos usar cualquiera sin problemas aunque nos quedamos con **Gini** por haber logrado ese pequeño extra de precisión final sobre el conjunto de prueba.

---
### 2.4 Poda Previa: Grid Search de Hiperparámetros

En este apartado vamos realizamos una búsqueda exhaustiva con **GridSearchCV** para encontrar la combinación ideal de hiperparámetros. No nos limitamos solo a la profundidad máxima sino que también ajustamos el número mínimo de muestras necesarias para dividir un nodo y el mínimo de muestras permitidas en cada hoja.

También exploramos la limitación del número máximo de nodos hoja para simplificar todavía más la estructura. El objetivo principal es aplicar técnicas de **poda previa** que detengan el crecimiento del árbol antes de que empiece a memorizar el ruido de los datos. Al final evaluamos el modelo ganador sobre el conjunto de test para comprobar si esta optimización logra superar los resultados de los experimentos anteriores.

In [ ]:
param_grid = {
    'max_depth':         [3, 5, 10, None],
    'min_samples_split': [2, 5, 10, 20],
    'min_samples_leaf':  [1, 2, 5, 10],
    'max_leaf_nodes':    [None, 10, 20, 50]
}

grid_tree = GridSearchCV(
    DecisionTreeClassifier(random_state=42),
    param_grid,
    cv=5,
    scoring='accuracy',
    n_jobs=-1
)

grid_tree.fit(X_train, y_train)

print(f"Mejores parámetros: {grid_tree.best_params_}")
print(f"Mejor CV score:     {grid_tree.best_score_:.4f}")
print(f"Test score:         {grid_tree.best_estimator_.score(X_test, y_test):.4f}")

Tabla con las mejores configuraciones encontradas por el GridSearch.

In [ ]:
results_tree = pd.DataFrame(grid_tree.cv_results_)
results_tree = results_tree.sort_values('mean_test_score', ascending=False)

rows = []
for _, r in results_tree.head(15).iterrows():
    tree_temp = DecisionTreeClassifier(
        max_depth=r['param_max_depth'],
        min_samples_split=r['param_min_samples_split'],
        min_samples_leaf=r['param_min_samples_leaf'],
        max_leaf_nodes=r['param_max_leaf_nodes'],
        random_state=42
    )
    tree_temp.fit(X_train, y_train)
    rows.append({
        'max_depth': str(r['param_max_depth']),
        'min_samples_split': r['param_min_samples_split'],
        'min_samples_leaf': r['param_min_samples_leaf'],
        'max_leaf_nodes': str(r['param_max_leaf_nodes']),
        'CV Acc': f"{r['mean_test_score']:.4f}",
        'Test Acc': f"{tree_temp.score(X_test, y_test):.4f}"
    })
pd.DataFrame(rows)

### Análisis de Resultados: Grid Search y Poda Previa

El proceso de búsqueda ha dado con una configuración muy equilibrada que soluciona los problemas de sobreajuste que vimos anteriormente.

La combinación ganadora utiliza un **max_depth de 5** junto a un límite de **20 nodos hoja**. Estos valores junto a un **min_samples_split de 10** consiguen un modelo mucho más robusto que el árbol infinito del principio. La estructura ahora es lo suficientemente profunda para entender el problema pero lo bastante sencilla para no perderse en detalles irrelevantes.

En cuanto a las métricas la precisión en validación cruzada ha subido hasta el **0.8147** mientras que en el test mantenemos un sólido **0.7989**. Lo más positivo de este resultado es que la brecha entre entrenamiento y prueba se ha reducido drásticamente y eso confirma que el modelo ya no está memorizando ruido sino aprendiendo reglas reales que funcionan con datos nuevos.

Si observamos la tabla vemos que muchas variantes con profundidad 5 dan resultados muy estables por encima del 0.81 en CV. Esto demuestra que el rendimiento del árbol se vuelve mucho más consistente una vez que aplicamos técnicas de poda previa para controlar su complejidad.

**Conclusión**:
Esta configuración final es la más fiable para un árbol de decisión en este dataset. Hemos logrado un equilibrio casi perfecto entre la capacidad de acierto y la simplicidad del modelo superando claramente al modelo sin restricciones.

---
### 2.5 Poda Posterior: Cost-Complexity Pruning

En este apartado vamos a probar una estrategia diferente para controlar el tamaño del árbol mediante la técnica de poda posterior. A diferencia de los métodos anteriores donde deteníamos el crecimiento antes de tiempo aquí dejamos que el árbol se desarrolle por completo y después vamos eliminando las ramas que menos información aportan al modelo.

Usamos el parámetro **ccp_alpha** para encontrar el equilibrio ideal entre la complejidad estructural y la capacidad de acierto. Generamos una secuencia de valores de alpha posibles y evaluamos cómo influye cada uno en la **validación cruzada** y en el conjunto de **test**. El objetivo es identificar el punto exacto donde el árbol se simplifica reduciendo su **número de hojas** sin que la precisión se vea perjudicada.

In [ ]:
# Obtener la secuencia de alphas generada por CART
path = DecisionTreeClassifier(random_state=42).cost_complexity_pruning_path(X_train, y_train)
ccp_alphas = path.ccp_alphas[:-1]  # El último genera solo el nodo raíz

for alpha in ccp_alphas:
    tree = DecisionTreeClassifier(ccp_alpha=alpha, random_state=42)
    cv   = cross_val_score(tree, X_train, y_train, cv=5)
    tree.fit(X_train, y_train)
    print(f"alpha={alpha:.5f}: CV={cv.mean():.4f}, "
          f"Test={tree.score(X_test, y_test):.4f}, hojas={tree.get_n_leaves()}")

Tabla con valores representativos del rango de ccp_alpha, incluyendo el mejor.

In [ ]:
rows_ccp = []
for alpha in ccp_alphas:
    tree = DecisionTreeClassifier(ccp_alpha=alpha, random_state=42)
    cv = cross_val_score(tree, X_train, y_train, cv=5)
    tree.fit(X_train, y_train)
    rows_ccp.append({
        'ccp_alpha': f"{alpha:.5f}",
        'cv_score': cv.mean(),
        'CV Accuracy': f"{cv.mean():.4f}",
        'Test Accuracy': f"{tree.score(X_test, y_test):.4f}",
        'Profundidad': tree.get_depth(),
        'N° Hojas': tree.get_n_leaves()
    })

df_ccp = pd.DataFrame(rows_ccp)
best_ccp_idx = df_ccp['cv_score'].idxmax()

# Seleccionar valores representativos: primero, 25%, 50%, mejor, 75%, último
indices = sorted(set([0, len(df_ccp)//4, len(df_ccp)//2, best_ccp_idx, 3*len(df_ccp)//4, len(df_ccp)-1]))
df_ccp.iloc[indices].drop(columns='cv_score')

### Análisis de Resultados: Cost-Complexity Pruning

La secuencia de valores de alpha muestra una transición muy clara desde un modelo sobreajustado hasta uno excesivamente simple.

* **Evolución de la complejidad**:
A medida que aumentamos el valor de alpha el número de hojas cae de forma drástica y el árbol se vuelve mucho más manejable. Al principio con un alpha de cero tenemos **159 hojas** y una profundidad de 20 niveles pero conforme el parámetro crece la estructura se va limpiando de ramas poco informativas.

* **Punto de máximo acierto en test**:
El rendimiento más alto para el conjunto de prueba se alcanza con valores de alpha alrededor de **0.00117**. En ese punto el modelo logra una precisión del **0.8212** con **100 hojas**. Este resultado es mejor que el del árbol original sin restricciones y confirma que eliminar el ruido mediante la poda posterior mejora significativamente la capacidad de generalización.

* **El equilibrio de la validación cruzada**:
Si nos fijamos en la validación cruzada el mejor registro aparece con un alpha de **0.00702**. Este valor genera un árbol muy compacto de solo **6 hojas** y profundidad 3 que mantiene un **0.8077** en validación. Aunque el test baja ligeramente a 0.7877 esta configuración es la más robusta porque evita cualquier riesgo de memorización innecesaria.

**Conclusión**:
La poda posterior ha demostrado ser una herramienta muy potente para optimizar el árbol. Los datos indican que valores de alpha pequeños logran exprimir al máximo la precisión en test mientras que valores un poco más altos como 0.007 ofrecen un modelo extremadamente simplificado y fiable.

---
### 2.6 Importancia de Características

En este último análisis del árbol vamos a descubrir cuáles son las variables que más han influido en las decisiones del modelo.

Utilizamos el atributo **feature_importances_** del mejor clasificador que encontramos en el paso anterior para cuantificar el peso de cada característica. El cálculo se basa en la **impureza de Gini** y nos permite ver qué datos han sido clave para separar a los supervivientes de los fallecidos.

Extraemos los nombres de las variables directamente desde el **preprocesador** para que coincidan con las transformaciones que aplicamos al principio. Después las ordenamos de mayor a menor relevancia para identificar de un vistazo los factores determinantes en el conjunto de datos.

In [ ]:
best_tree = grid_tree.best_estimator_
importances  = best_tree.feature_importances_
# Adaptado: usamos get_feature_names_out() porque el ColumnTransformer cambia los nombres
feature_names = preprocessor.get_feature_names_out().tolist()

indices = importances.argsort()[::-1]
for i, idx in enumerate(indices):
    print(f"  {i+1}. {feature_names[idx]}: {importances[idx]:.4f}")

Tabla con: Ranking | Feature | Importancia (Gini).

In [ ]:
rows = []
for i, idx in enumerate(indices):
    rows.append({
        'Ranking': i + 1,
        'Feature': feature_names[idx],
        'Importancia (Gini)': f"{importances[idx]:.4f}"
    })
pd.DataFrame(rows)

### Análisis de Resultados: Importancia de Características

El ranking de importancia nos permite entender qué variables está priorizando el árbol para tomar sus decisiones.

* **El sexo domina el modelo**
Ser hombre o mujer es la característica más relevante con una importancia de **0.5437**. Representa más de la mitad del peso total de todas las variables y eso demuestra que el género fue el principal filtro de supervivencia para este algoritmo.

* **Clase y edad como factores secundarios**
La **tercera clase** con un **0.1356** y la **edad** con un **0.1318** ocupan los siguientes puestos en relevancia. Ambos factores tienen un impacto casi idéntico y ayudan al modelo a refinar las predicciones después de haber filtrado primero por el sexo del pasajero.

* **Variables con impacto nulo**
Es muy interesante observar que el puerto de embarque en **Queenstown** y el número de padres o hijos a bordo (**Parch**) tienen una importancia de **0.0000**. El árbol ha decidido ignorar estos datos por completo porque no aportan información útil para mejorar la clasificación final una vez aplicados los filtros anteriores.

**Conclusión final**
El modelo se apoya fundamentalmente en tres pilares que son el **sexo**, la **clase** y la **edad**. El resto de características como el precio del billete o el tamaño de la familia apenas aportan matices pequeños al resultado del clasificador.

---
### 2.7 Visualización del Árbol

En esta parte vamos a dibujar la estructura del mejor árbol que hemos obtenido para entender cómo se ramifican las decisiones.

Para que el gráfico sea legible limitamos la visualización a una **profundidad de 4 niveles**. Esto permite seguir el camino que recorre un pasajero desde la raíz hasta las hojas sin que el dibujo se vuelva caótico. Además de la imagen generamos una **representación en texto** que detalla cada condición de división y el valor de impureza en cada nodo.

Es la mejor forma de comprobar visualmente cómo el **sexo** y la **clase** van segmentando el dataset hasta llegar a la predicción final de supervivencia.

In [ ]:
from sklearn.tree import plot_tree, export_text

class_names = ['No Survived', 'Survived']

plt.figure(figsize=(20, 10))
plot_tree(best_tree, feature_names=feature_names, class_names=class_names,
          filled=True, rounded=True, fontsize=10, max_depth=4)
plt.savefig('images/decision_tree.png', dpi=300, bbox_inches='tight')
plt.show()

# Representación en texto
print(export_text(best_tree, feature_names=feature_names, max_depth=4))

---
### 2.8 Evaluación Final (Parte 2)

En esta última parte vamos a evaluar el rendimiento del mejor árbol de decisión mediante el **classification report** y la **matriz de confusión**. 

También generamos una tabla comparativa para enfrentar las tres estrategias de construcción que hemos visto. Comparamos el modelo **sin restricciones** contra las versiones optimizadas con **poda previa** y **poda posterior** para ver cuál logra la mejor precisión final y el **F1-Score** más equilibrado. Es la forma definitiva de comprobar si la simplificación del árbol realmente ayuda a generalizar mejor con datos nuevos.

In [ ]:
# Mejor modelo de Parte 2: el del GridSearch (poda previa)
best_tree_model = grid_tree.best_estimator_
best_tree_model.fit(X_train, y_train)
y_pred_tree = best_tree_model.predict(X_test)

print(classification_report(y_test, y_pred_tree))

cm = confusion_matrix(y_test, y_pred_tree)
ConfusionMatrixDisplay(confusion_matrix=cm).plot()
plt.title('Matriz de Confusión - Mejor Árbol de Decisión')
plt.savefig('images/confusion_matrix_tree.png', dpi=300, bbox_inches='tight')
plt.show()

Tabla comparativa final de la Parte 2: sin restricciones, mejor poda previa y mejor poda posterior.

In [ ]:
# Encontrar el mejor alpha de la poda posterior
best_alpha = df_ccp.loc[best_ccp_idx, 'ccp_alpha']
best_post_tree = DecisionTreeClassifier(ccp_alpha=float(best_alpha), random_state=42)
best_post_tree.fit(X_train, y_train)
cv_post = cross_val_score(best_post_tree, X_train, y_train, cv=5)
yp_post = best_post_tree.predict(X_test)

# Sin restricciones
tree_full2 = DecisionTreeClassifier(random_state=42)
tree_full2.fit(X_train, y_train)
cv_full = cross_val_score(tree_full2, X_train, y_train, cv=5)
yp_full = tree_full2.predict(X_test)

# Mejor poda previa (GridSearch)
best_pre = grid_tree.best_estimator_
best_pre.fit(X_train, y_train)
cv_pre = cross_val_score(best_pre, X_train, y_train, cv=5)
yp_pre = best_pre.predict(X_test)

rows = []
for name, model, cv, yp in [
        ('Sin restricciones', tree_full2, cv_full, yp_full),
        ('Mejor poda previa', best_pre, cv_pre, yp_pre),
        ('Mejor poda posterior', best_post_tree, cv_post, yp_post)]:
    rows.append({
        'Modelo': name,
        'CV Accuracy': f"{cv.mean():.4f}",
        'Test Accuracy': f"{accuracy_score(y_test, yp):.4f}",
        'F1-Score': f"{f1_score(y_test, yp):.4f}",
        'Profundidad': model.get_depth(),
        'N° Hojas': model.get_n_leaves()
    })
pd.DataFrame(rows)

### Análisis de Resultados: Evaluación Final del Árbol

El mejor árbol de decisión alcanza un **80% de precisión** general en el test y eso lo sitúa a la par de los modelos SVM que vimos antes.

Si analizamos la matriz de confusión vemos que el modelo es extremadamente eficaz detectando a los pasajeros que no sobrevivieron con un **recall de 0.94**. El punto débil está en la clase de los supervivientes porque solo es capaz de identificar correctamente al **58%** de ellos. Esto nos indica que el árbol prefiere ser conservador y clasificar a los casos dudosos como fallecidos para asegurar su precisión.

En la tabla comparativa queda claro que las técnicas de poda han funcionado muy bien. La **Mejor poda previa** es la más equilibrada con un **0.8147** en validación cruzada y una estructura de solo **20 hojas**. Por otro lado la **poda posterior** ha logrado simplificar el modelo al máximo con apenas **6 hojas** manteniendo una precisión muy competitiva del **0.7877**. El modelo **Sin restricciones** tiene un F1-Score ligeramente superior pero sus **159 hojas** lo convierten en una estructura demasiado ruidosa para ser útil en la práctica.

**Conclusión final**:
Los resultados demuestran que no hace falta un árbol gigante para predecir bien los datos del Titanic. Un modelo podado con **20 hojas** es mucho más robusto y fácil de interpretar que el árbol original y además ofrece una confianza mucho mayor en la validación cruzada.

---
## Parte 3: Random Forest

### 3.1 Random Forest con Configuración por Defecto

En este apartado vemos los modelos de ensamble con el algoritmo **Random Forest**. En lugar de usar un único árbol entrenamos un conjunto de **100 árboles de decisión** independientes para que voten el resultado final y así reducir la varianza.

Una de las ventajas de este modelo es que podemos calcular el **OOB Score** que nos da una estimación del acierto sin necesidad de usar un conjunto de validación externo. Al final comparamos esa métrica con el **Test Score** para comprobar si el bosque está generalizando bien o si sufre de sobreajuste al combinar tantos árboles distintos.

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1, oob_score=True)
rf.fit(X_train, y_train)

print(f"OOB Score:  {rf.oob_score_:.4f}")
print(f"Test Score: {rf.score(X_test, y_test):.4f}")

Tabla con: Configuración | OOB Score | Test Accuracy | F1-Score.

In [ ]:
y_pred_rf_default = rf.predict(X_test)
rows = [{
    'Configuración': 'RF por defecto (n=100)',
    'OOB Score': f"{rf.oob_score_:.4f}",
    'Test Accuracy': f"{rf.score(X_test, y_test):.4f}",
    'F1-Score': f"{f1_score(y_test, y_pred_rf_default):.4f}"
}]
pd.DataFrame(rows)

### Análisis de Resultados: Random Forest por Defecto

Los números que arroja el primer bosque aleatorio demuestran la gran estabilidad de este algoritmo frente a los árboles individuales.

El **OOB Score de 0.8048** es prácticamente idéntico al **0.7989** que hemos obtenido en el conjunto de prueba. Esto es una señal muy positiva porque confirma que la estimación interna del modelo es totalmente fiable y que el bosque está generalizando de forma correcta sin caer en un sobreajuste excesivo.

Con un **F1-Score de 0.7273** el modelo ya se sitúa en niveles muy competitivos superando los resultados de los árboles individuales optimizados. Al promediar las predicciones de **100 árboles** distintos logramos filtrar gran parte del ruido que antes podía confundir al clasificador.

**Conclusión**:
El Random Forest con su configuración base ya se comporta como un modelo muy sólido y equilibrado. La coherencia entre la validación interna y el test nos da una base muy segura para intentar mejorar estas cifras ajustando los hiperparámetros en el siguiente paso.

---
### 3.2 Efecto del Número de Estimadores

En esta sección vamos a analizar cómo influye la cantidad de árboles en el rendimiento global del **Random Forest**.

Probamos diferentes valores para el parámetro **n_estimators** empezando por un bosque muy pequeño de solo 10 árboles hasta llegar a uno más robusto con 500. El objetivo es observar si aumentar el número de estimadores mejora siempre la precisión o si llega un punto donde el modelo se estabiliza y ya no aporta ninguna ventaja extra. Evaluamos tanto el **OOB Score** como el **Test Accuracy** para encontrar el tamaño de bosque más eficiente.

In [ ]:
for n in [10, 25, 50, 100, 200, 500]:
    rf = RandomForestClassifier(n_estimators=n, random_state=42, n_jobs=-1, oob_score=True)
    rf.fit(X_train, y_train)
    print(f"n_estimators={n:>3}: OOB={rf.oob_score_:.4f}, "
          f"Test={rf.score(X_test, y_test):.4f}")

Tabla con: n_estimators | OOB Score | Test Accuracy | Tiempo (s).

In [ ]:
rows = []
for n in [10, 25, 50, 100, 200, 500]:
    rf = RandomForestClassifier(n_estimators=n, random_state=42, n_jobs=-1, oob_score=True)
    t0 = time()
    rf.fit(X_train, y_train)
    t1 = time()
    rows.append({
        'n_estimators': n,
        'OOB Score': f"{rf.oob_score_:.4f}",
        'Test Accuracy': f"{rf.score(X_test, y_test):.4f}",
        'Tiempo (s)': f"{t1-t0:.3f}"
    })
pd.DataFrame(rows)

### Análisis de Resultados: Efecto del Número de Estimadores

Los datos muestran una tendencia muy clara hacia la mejora de la estabilidad a medida que añadimos más árboles al bosque.

Con solo **10 estimadores** el modelo se queda corto con un **OOB de 0.7725**. Es una cifra bastante baja que indica que el ensamble no tiene suficientes votos para filtrar el ruido de los datos de forma eficaz.

Entre los **25 y los 100 árboles** el rendimiento se mantiene totalmente estable en un **0.7989** de acierto en test. Es interesante ver cómo el modelo no mejora en este tramo pero el tiempo de ejecución sube de forma lineal conforme aumentamos la complejidad.

El salto de calidad definitivo llega al alcanzar los **200 estimadores** donde la precisión en test sube hasta el **0.8156**. Al mantener este mismo acierto con **500 árboles** confirmamos que el modelo ha llegado a su techo de aprendizaje para esta configuración. El **OOB Score** también alcanza su valor máximo con un **0.8132** y eso nos asegura que el bosque es extremadamente robusto.

**Conclusión**:
Aumentar el número de árboles ha servido para ganar un **1.6%** extra de precisión respecto a la configuración por defecto. Aunque el tiempo de entrenamiento sube hasta superar el segundo completo merece la pena usar al menos **200 estimadores** para garantizar el mejor resultado posible.

---
### 3.3 Efecto de max_features

En este apartado vamos a analizar el comportamiento del bosque según el parámetro **max_features**. Este valor controla cuántas variables tiene permitido consultar cada árbol en cada división del nodo.

Probamos varias opciones como la raíz cuadrada o el logaritmo del total de variables junto a un porcentaje fijo o el uso de todas las características a la vez. El objetivo de este experimento es encontrar el equilibrio perfecto entre la fuerza individual de cada árbol y la diversidad del conjunto para mejorar la precisión final. Comparamos los resultados de **OOB Score** y de **Test Accuracy** para decidir qué configuración es la más robusta.

In [ ]:
for mf in ['sqrt', 'log2', 0.5, None]:
    rf = RandomForestClassifier(n_estimators=100, max_features=mf,
                                random_state=42, n_jobs=-1, oob_score=True)
    rf.fit(X_train, y_train)
    print(f"max_features={str(mf):>6}: OOB={rf.oob_score_:.4f}, "
          f"Test={rf.score(X_test, y_test):.4f}")

Tabla que pide el enunciado: max_features | OOB Score | Test Accuracy.

In [ ]:
rows = []
for mf in ['sqrt', 'log2', 0.5, None]:
    rf = RandomForestClassifier(n_estimators=100, max_features=mf,
                                random_state=42, n_jobs=-1, oob_score=True)
    rf.fit(X_train, y_train)
    rows.append({
        'max_features': 'None (todas)' if mf is None else str(mf),
        'OOB Score': f"{rf.oob_score_:.4f}",
        'Test Accuracy': f"{rf.score(X_test, y_test):.4f}"
    })
pd.DataFrame(rows)

### Análisis de Resultados sobre max_features

Los resultados muestran que limitar demasiado la cantidad de variables en cada nodo perjudica el rendimiento del bosque.

Las opciones de **sqrt** y **log2** devuelven exactamente los mismos valores con un acierto en test del **0.7989**. Esto ocurre porque en datasets pequeños como el del Titanic ambas fórmulas suelen dar un número de características muy similar y eso acaba generando árboles casi idénticos.

Al permitir que el modelo consulte el **50% de las variables** la precisión sube hasta el **0.8045**. Sin embargo el mejor resultado aparece al usar **todas las características** con la opción **None**. En este caso el modelo alcanza un **0.8156** de acierto en test y un **0.8188** en el OOB Score. Esto nos indica que para este problema concreto los árboles funcionan mejor cuando tienen toda la información disponible para decidir cada división.

**Conclusión**:
A diferencia de lo que ocurre en otros problemas aquí no necesitamos forzar tanto la aleatoriedad para evitar el sobreajuste. La configuración que usa todas las variables es la más potente y nos permite conseguir la mejor métrica de toda la comparativa de este apartado.

---
### 3.4 Grid Search

En este paso vamos a realizar una búsqueda exhaustiva con **GridSearchCV** para encontrar la combinación definitiva de hiperparámetros. No nos conformamos con ajustar los parámetros por separado sino que exploramos cómo interactúan el número de árboles con la profundidad y el número de muestras por hoja.

Buscamos el modelo más equilibrado que sea capaz de generalizar correctamente sobre los datos del Titanic. Al final calculamos el **CV score** y la precisión sobre el **conjunto de test** para confirmar si esta optimización ha dado sus frutos.

In [ ]:
param_grid = {
    'n_estimators':     [100, 200],
    'max_features':     ['sqrt', 'log2', None],
    'max_depth':        [None, 10, 20],
    'min_samples_leaf': [1, 2, 5]
}

grid_rf = GridSearchCV(
    RandomForestClassifier(random_state=42, n_jobs=-1),
    param_grid,
    cv=5,
    scoring='accuracy',
    n_jobs=-1
)

grid_rf.fit(X_train, y_train)

print(f"Mejores parámetros: {grid_rf.best_params_}")
print(f"Mejor CV score:     {grid_rf.best_score_:.4f}")
print(f"Test score:         {grid_rf.best_estimator_.score(X_test, y_test):.4f}")

### Análisis de Resultados: Optimización del Random Forest

La búsqueda ha finalizado con una configuración que prioriza la robustez del bosque sobre la complejidad individual de cada árbol. 

El modelo ganador utiliza un **max_depth de 10** y un **min_samples_leaf de 2**. Estos límites aseguran que los árboles no crezcan de forma descontrolada y evitan el sobreajuste que sufríamos con el árbol de decisión básico. Además el uso de **200 estimadores** confirma que un bosque más grande ayuda a estabilizar las predicciones y reduce la varianza del error.

Es muy interesante observar que el parámetro **max_features=sqrt** ha resultado ser el mejor en esta búsqueda conjunta. Aunque en las pruebas individuales parecía que usar todas las variables era mejor la combinación con el resto de hiperparámetros demuestra que fomentar la diversidad entre los árboles es la clave para ganar precisión.

Las métricas finales son excelentes porque el **CV score alcanza el 0.8287** y eso lo sitúa como uno de los modelos más potentes de toda la práctica. La precisión en el conjunto de test se queda en un **0.8045** lo que representa un rendimiento muy sólido y equilibrado para enfrentarse a datos nuevos.

**Conclusión**:
El **Random Forest** optimizado es el modelo más fiable de los analizados hasta ahora. Ha conseguido batir al árbol de decisión simple y se muestra mucho más consistente que los modelos SVM gracias a la combinación de múltiples votos.

---
### 3.5 Importancia de Características

En este apartado vamos a comparar dos formas distintas de medir qué variables son las más importantes para el **Random Forest**.

Primero calculamos la **importancia de Gini** que es la métrica que el modelo genera por defecto basándose en la reducción de la impureza en cada nodo. Después aplicamos la **Permutation Importance** que suele ser un método mucho más fiable cuando existen variables correlacionadas entre sí. Esta técnica consiste en desordenar aleatoriamente una columna para ver cuánto cae la precisión del modelo y eso nos da una visión más realista del peso real de cada factor.

Al final comparamos ambas listas para comprobar si el ranking de variables se mantiene estable o si alguna métrica nos da una sorpresa sobre la relevancia de los datos.

In [ ]:
import numpy as np
from sklearn.inspection import permutation_importance

best_rf = grid_rf.best_estimator_

# Gini Importance (MDI)
importances_gini = best_rf.feature_importances_

# Permutation Importance (más fiable con features correlacionadas)
perm = permutation_importance(best_rf, X_test, y_test, n_repeats=10, random_state=42)

indices = importances_gini.argsort()[::-1]
for i, idx in enumerate(indices):
    print(f"  {i+1}. {feature_names[idx]}: "
          f"Gini={importances_gini[idx]:.4f}, "
          f"Perm={perm.importances_mean[idx]:.4f} (+/-{perm.importances_std[idx]:.4f})")

Tabla con: Ranking | Feature | Gini Importance | Permutation Importance.

In [ ]:
rows = []
for i, idx in enumerate(indices):
    rows.append({
        'Ranking': i + 1,
        'Feature': feature_names[idx],
        'Gini Importance': f"{importances_gini[idx]:.4f}",
        'Permutation Importance': f"{perm.importances_mean[idx]:.4f} (+/-{perm.importances_std[idx]:.4f})"
    })
pd.DataFrame(rows)

### Análisis de Resultados: Gini vs. Permutación

Comparar ambas métricas nos permite ver qué variables son realmente útiles y cuáles están engañando un poco al modelo.

El **sexo del pasajero** vuelve a coronarse como el factor más decisivo en ambas listas. Con una importancia de Gini de **0.3469** y una permutación de **0.1693** queda claro que es la variable que más peso tiene en el acierto final del bosque.

Sin embargo el precio del billete (**Fare**) presenta una discrepancia muy curiosa. Gini le otorga un valor muy alto del **0.2254** pero la permutación lo desploma hasta un **0.0240**. Esto sucede porque la importancia de Gini tiende a inflar el peso de las variables numéricas con muchos valores únicos mientras que la permutación nos dice que el precio no es tan vital para la predicción real como parece a simple vista.

La **edad** y la **tercera clase** se mantienen como valores seguros. Ambas variables muestran puntuaciones sólidas en los dos métodos y eso las confirma como los mejores predictores después del género.

Por último es llamativo que variables como **SibSp** o **Pclass_2** tengan valores de permutación negativos o nulos. Esto significa que esas características apenas aportan nada al modelo final e incluso podrían estar actuando como ruido en lugar de ayudar a clasificar mejor.

**Conclusión**:
Para este Random Forest debemos fiarnos más del ranking de **permutación**. Nos revela que una vez que conocemos el sexo la clase y la edad el resto de variables apenas mueven la aguja del rendimiento del modelo.

---
### 3.6 Extra Trees

En este último apartado de los ensambles vamos a probar el algoritmo de **Extra Trees**. Este modelo es una variante todavía más aleatoria que el Random Forest convencional porque no busca el mejor punto de corte posible para cada variable sino que elige umbrales de división de forma totalmente aleatoria.

Entrenamos un bosque de **100 árboles** con esta configuración para ver si ese extra de aleatoriedad ayuda a reducir todavía más el sobreajuste en los datos del Titanic. Calculamos la precisión en **validación cruzada** y sobre el **conjunto de prueba** para comparar su rendimiento final contra el Random Forest y el resto de modelos de la práctica.

In [ ]:
from sklearn.ensemble import ExtraTreesClassifier

et = ExtraTreesClassifier(n_estimators=100, random_state=42, n_jobs=-1)
et.fit(X_train, y_train)
cv_et = cross_val_score(et, X_train, y_train, cv=5)

print(f"Extra Trees: CV={cv_et.mean():.4f} (+/-{cv_et.std():.4f}), "
      f"Test={et.score(X_test, y_test):.4f}")

### Análisis de Resultados: Extra Trees

Los resultados de este modelo nos dejan una conclusión bastante curiosa y sorprendente.

La precisión en validación cruzada se queda en un **0.7740**. Esta cifra es inferior a la que obtuvimos con el Random Forest y eso sugiere que la aleatoriedad extrema al elegir los puntos de corte puede estar perjudicando la estabilidad del modelo en diferentes subconjuntos. Sin embargo el acierto en el conjunto de test se dispara hasta alcanzar el **0.8268**.

Estamos ante el mejor resultado sobre el conjunto de prueba hasta ahora. Supera incluso al SVM y al Random Forest optimizado aunque debemos tomar esta cifra con cautela debido a la brecha que existe respecto a la validación cruzada. Parece que la simplificación radical de los umbrales de decisión ha funcionado especialmente bien con las muestras específicas que componen nuestro set de test.

**Conclusión**:
**Extra Trees** se convierte en el ganador inesperado en cuanto a precisión pura sobre el test final. Aunque su validación es algo más inestable su capacidad para reducir la varianza mediante la aleatoriedad total lo convierte en una alternativa muy potente frente a los bosques tradicionales para este dataset.

---
### 3.7 Evaluación Final (Parte 3)
Apartado 3.7 del enunciado. Classification report y matriz de confusión del mejor modelo, más tabla comparativa.

### 3.7 Evaluación Final y Comparativa de Ensambles

En este paso final del bloque enfrentamos a todos los modelos de ensamble contra el árbol base para sacar conclusiones definitivas.

Primero analizamos la **matriz de confusión** del mejor **Random Forest** para ver cómo se distribuyen los aciertos y errores. Después utilizamos la tabla comparativa para medir el impacto real de la optimización. Es aquí donde comprobamos si el esfuerzo de buscar los mejores hiperparámetros compensa el tiempo de ejecución extra frente a las versiones por defecto o al algoritmo de **Extra Trees**.

La comparativa nos muestra dos realidades distintas. El **Random Forest optimizado** es el modelo más fiable y estable con un **CV Accuracy de 0.8287** mientras que el **Extra Trees** se comporta como un "sprinter" que logra la cifra más alta en el conjunto de **test** con un **0.8268**.

**Resumen de la tabla:**
* **Estabilidad:** El Random Forest tras el Grid Search es el que mejor generaliza según la validación cruzada.
* **Precisión pura:** Extra Trees lidera el test a pesar de tener una validación más inestable.
* **Eficiencia:** El árbol base es instantáneo pero queda muy lejos en capacidad de acierto respecto a sus hermanos mayores.

In [ ]:
best_rf_model = grid_rf.best_estimator_
best_rf_model.fit(X_train, y_train)
y_pred_rf = best_rf_model.predict(X_test)

print(classification_report(y_test, y_pred_rf))

cm = confusion_matrix(y_test, y_pred_rf)
ConfusionMatrixDisplay(confusion_matrix=cm).plot()
plt.title('Matriz de Confusión - Mejor Random Forest')
plt.savefig('images/confusion_matrix_rf.png', dpi=300, bbox_inches='tight')
plt.show()

Tabla comparativa final de la Parte 3.

In [ ]:
# Decision Tree baseline (sin restricciones)
dt_base = DecisionTreeClassifier(random_state=42)
dt_base.fit(X_train, y_train)
cv_dt = cross_val_score(dt_base, X_train, y_train, cv=5)
yp_dt = dt_base.predict(X_test)

# RF por defecto
rf_def = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1, oob_score=True)
t0 = time()
rf_def.fit(X_train, y_train)
t_rf_def = time() - t0
cv_rf_def = cross_val_score(rf_def, X_train, y_train, cv=5)
yp_rf_def = rf_def.predict(X_test)

# RF mejor config
best_rf2 = grid_rf.best_estimator_
t0 = time()
best_rf2.fit(X_train, y_train)
t_rf_best = time() - t0
cv_rf_best = cross_val_score(best_rf2, X_train, y_train, cv=5)
yp_rf_best = best_rf2.predict(X_test)

# Extra Trees
et2 = ExtraTreesClassifier(n_estimators=100, random_state=42, n_jobs=-1)
t0 = time()
et2.fit(X_train, y_train)
t_et = time() - t0
cv_et2 = cross_val_score(et2, X_train, y_train, cv=5)
yp_et = et2.predict(X_test)

rows = []
for name, cv, yp, oob, t in [
        ('Decision Tree (baseline)', cv_dt, yp_dt, 'N/A', 0),
        ('Random Forest por defecto', cv_rf_def, yp_rf_def, f"{rf_def.oob_score_:.4f}", t_rf_def),
        ('Random Forest mejor config', cv_rf_best, yp_rf_best, f"{best_rf2.oob_score_:.4f}" if hasattr(best_rf2, "oob_score_") else "N/A", t_rf_best),
        ('Extra Trees', cv_et2, yp_et, 'N/A', t_et)]:
    rows.append({
        'Modelo': name,
        'CV Accuracy': f"{cv.mean():.4f}",
        'Test Accuracy': f"{accuracy_score(y_test, yp):.4f}",
        'F1-Score': f"{f1_score(y_test, yp):.4f}",
        'OOB Score': oob,
        'Tiempo (s)': f"{t:.3f}"
    })
pd.DataFrame(rows)

### Análisis de Resultados: Comparativa de Ensambles

Al observar los datos finales queda claro que la potencia de los ensambles ha elevado el nivel de las predicciones respecto al árbol individual.

* **Rendimiento del Mejor Random Forest**:
La matriz de confusión revela un modelo muy equilibrado con una precisión global del **80%**. Lo más destacable es su capacidad para identificar a los no supervivientes con un **recall de 0.89** (98 aciertos). En el caso de los supervivientes el modelo es algo más exigente logrando identificar correctamente al **67%** de ellos (46 aciertos) pero manteniendo una precisión alta del **0.79**.

* **Duelo de Modelos: RF vs. Extra Trees**:
La tabla comparativa nos muestra una competencia muy reñida. El **Random Forest optimizado** es el modelo más fiable de todos en términos de consistencia con el mejor **CV Accuracy (0.8287)** de toda la práctica. Sin embargo el modelo de **Extra Trees** vuelve a dar la sorpresa al conseguir el **Test Accuracy más alto con un 0.8268** y el mejor **F1-Score (0.7669)**.

* **Conclusión de los Ensambles**:
Si buscamos robustez y confianza en los resultados la configuración de **Random Forest** ganada por el Grid Search es la opción lógica. Pero si el objetivo es maximizar el acierto puro en datos nunca vistos el algoritmo de **Extra Trees** se convierte en el modelo con mejor desempeño sobre el test final para este dataset del Titanic.

**Resumen rápido**:
Hemos pasado de un árbol base algo inestable a modelos de ensamble que superan con comodidad el **80%** de acierto reduciendo drásticamente el error y mejorando la generalización.

---
## Parte 4: Gradient Boosting

### 4.1 Gradient Boosting con Configuración por Defecto

En este paso probamos el **Gradient Boosting**. Es un modelo que también utiliza muchos árboles de decisión pero los construye de una forma diferente a los anteriores. En lugar de crear todos los árboles a la vez de forma independiente cada nuevo árbol se fija en los fallos que cometieron los de atrás para intentar corregirlos.

Entrenamos el modelo con sus opciones de serie para ver qué tal funciona sin ajustar nada todavía. Usamos la **validación cruzada** para comprobar si el aprendizaje es estable en diferentes partes del dataset y el **conjunto de test** para obtener la precisión final. El objetivo es ver si esta técnica de corregir errores poco a poco consigue mejores resultados que los métodos que hemos visto hasta ahora.

In [ ]:
from sklearn.ensemble import GradientBoostingClassifier

gb = GradientBoostingClassifier(random_state=42)
gb.fit(X_train, y_train)
cv_gb = cross_val_score(gb, X_train, y_train, cv=5)

print(f"CV={cv_gb.mean():.4f} (+/-{cv_gb.std():.4f}), "
      f"Test={gb.score(X_test, y_test):.4f}")

### Análisis de Resultados: Gradient Boosting Base

Los números de este modelo muestran un rendimiento muy alto en la validación pero un descenso en la prueba final.

La **validación cruzada de 0.8260** es una de las mejores puntuaciones obtenidas hasta ahora. Esto significa que el modelo es muy capaz de aprender los patrones del dataset de entrenamiento. Sin embargo la precisión en el **conjunto de test baja al 0.7877**.

Esta diferencia entre ambos valores indica que el modelo tiene un problema de sobreajuste. El Gradient Boosting por defecto crea árboles que se adaptan demasiado bien a los datos conocidos y pierden efectividad cuando ven datos nuevos. 

**Conclusión**:
El modelo es muy potente pero necesita ser modificado. Tendriamos que encontrar una configuración que equilibre mejor el aprendizaje para que el resultado en el test sea tan bueno como el de la validación.

---
### 4.2 Efecto del Número de Estimadores y Learning Rate

En esta sección analizamos la relación entre el número de árboles (**n_estimators**) y la velocidad de aprendizaje (**learning_rate**). Estos dos parámetros funcionan siempre en equipo. Si configuramos una velocidad de aprendizaje baja cada árbol aporta una corrección muy pequeña y por eso el modelo necesita más cantidad de árboles para funcionar bien. Si la velocidad es alta el modelo aprende más rápido pero es más fácil que termine sobreajustando los datos.

Probamos varias combinaciones para encontrar el equilibrio exacto. Queremos ver si es mejor dar muchos pasos pequeños o pocos pasos grandes para conseguir la mejor precisión tanto en la **validación cruzada** como en el **conjunto de test**.

In [ ]:
configs = [
    (50,  0.1), (100, 0.1), (200, 0.1),
    (50,  0.5), (100, 0.05), (200, 0.01)
]

for n_est, lr in configs:
    gb = GradientBoostingClassifier(n_estimators=n_est, learning_rate=lr, random_state=42)
    cv = cross_val_score(gb, X_train, y_train, cv=5)
    gb.fit(X_train, y_train)
    print(f"n={n_est}, lr={lr}: CV={cv.mean():.4f}, Test={gb.score(X_test, y_test):.4f}")

Tabla con: n_estimators | learning_rate | CV Accuracy | Test Accuracy.

In [ ]:
rows = []
for n_est, lr in configs:
    gb = GradientBoostingClassifier(n_estimators=n_est, learning_rate=lr, random_state=42)
    cv = cross_val_score(gb, X_train, y_train, cv=5)
    gb.fit(X_train, y_train)
    rows.append({
        'n_estimators': n_est,
        'learning_rate': lr,
        'CV Accuracy': f"{cv.mean():.4f}",
        'Test Accuracy': f"{gb.score(X_test, y_test):.4f}"
    })
pd.DataFrame(rows)

### Análisis de Resultados: Estimadores y Learning Rate

Los resultados confirman que la velocidad a la que el modelo aprende influye mucho en su capacidad para acertar con datos nuevos.

* **Efecto de la velocidad alta**:
Cuando usamos un **learning_rate de 0.5** con solo 50 árboles conseguimos un **0.8045** en el test. Es una cifra muy buena pero la validación cruzada es más baja que en otras pruebas. Esto pasa porque el modelo aprende muy rápido y puede saltarse detalles importantes por el camino.

* **Efecto de la velocidad baja**:
La combinación de **200 árboles** con un **learning_rate de 0.01** también logra un **0.8045** en el test. Esta opción es mucho más segura y estable porque el modelo da muchos pasos pequeños para corregir los errores. Al comparar esta prueba con la de velocidad alta vemos que llegamos al mismo acierto pero de una forma mucho más controlada.

* **El problema del punto medio**:
Curiosamente la configuración de 100 árboles con velocidad 0.1 es la que saca mejor nota en validación cruzada pero la peor en el test. Esto demuestra que a veces el modelo se confía demasiado al aprender de los datos de entrenamiento y luego falla al ver datos que no conoce.

**Conclusión**:
Los datos indican que para este problema es mejor usar una velocidad de aprendizaje baja con muchos árboles o una velocidad alta con pocos. Ambas estrategias ayudan a que el **Gradient Boosting** sea más fiable y no se pierda memorizando los datos de entrenamiento.

---
### 4.3 Efecto de max_depth

Vamos a ver cómo influye la profundidad de los árboles en el resultado final. En el **Gradient Boosting** los árboles suelen ser a propósito muy simples porque el modelo prefiere sumar muchos conocimientos pequeños en lugar de unos pocos muy complejos.

Probamos valores desde **1** nivel de profundidad hasta **5**. El objetivo es observar si aumentar la complejidad de cada árbol ayuda al modelo o si por el contrario provoca que se aprenda de memoria los datos de entrenamiento. Comparamos la precisión en **entrenamiento**, **validación** y **test** para identificar cuándo el modelo deja de ser útil para datos nuevos.


In [ ]:
for depth in [1, 2, 3, 5]:
    gb = GradientBoostingClassifier(max_depth=depth, n_estimators=100,
                                    learning_rate=0.1, random_state=42)
    cv = cross_val_score(gb, X_train, y_train, cv=5)
    gb.fit(X_train, y_train)
    print(f"max_depth={depth}: CV={cv.mean():.4f}, "
          f"train={gb.score(X_train, y_train):.4f}, test={gb.score(X_test, y_test):.4f}")

Tabla con: max_depth | Train Acc | CV Accuracy | Test Accuracy.

In [ ]:
rows = []
for depth in [1, 2, 3, 5]:
    gb = GradientBoostingClassifier(max_depth=depth, n_estimators=100,
                                    learning_rate=0.1, random_state=42)
    cv = cross_val_score(gb, X_train, y_train, cv=5)
    gb.fit(X_train, y_train)
    rows.append({
        'max_depth': f'{depth} (stump)' if depth == 1 else str(depth),
        'Train Acc': f"{gb.score(X_train, y_train):.4f}",
        'CV Accuracy': f"{cv.mean():.4f}",
        'Test Accuracy': f"{gb.score(X_test, y_test):.4f}"
    })
pd.DataFrame(rows)

### Análisis de Resultados: Profundidad del Árbol (max_depth)

Los datos indican que un aumento en la profundidad de los árboles no garantiza una mejora en la precisión final del modelo.

* **Resultados con profundidad baja**:
Con una profundidad de 2 el modelo alcanza el mejor resultado en el test con un valor de **0.8101**. Incluso con profundidad 1 el desempeño es alto llegando al **0.8045**. Esto demuestra que el Gradient Boosting funciona de forma más eficaz cuando combina muchos árboles sencillos.

* **Sobreajuste en profundidades altas**:
Al subir a profundidad 3 y 5 la precisión en el entrenamiento aumenta hasta alcanzar el **0.9593** pero el acierto en el test disminuye hasta el **0.7877**. Esto sucede porque el algoritmo aprende detalles específicos de los datos de entrenamiento que no se repiten en los datos de prueba.

**Conclusión**:
El valor de **2** es el parámetro más equilibrado para este caso. Consigue la mayor precisión en datos nuevos sin que el modelo aprenda de memoria los ejemplos del entrenamiento.

---
### 4.4 Grid Search

En este apartado utilizamos **GridSearchCV** para probar de forma conjunta los parámetros más importantes del modelo. Hasta ahora los hemos analizado por separado pero en esta búsqueda los combinamos para ver cómo interactúan entre sí.

Añadimos el parámetro **subsample** que sirve para que cada árbol utilice solo una parte del conjunto de datos. Esta técnica ayuda a que el algoritmo no se ajuste demasiado a los ejemplos de entrenamiento y mejore su capacidad de predicción en situaciones nuevas. El proceso termina calculando la precisión mediante **validación cruzada** y realizando una prueba final sobre el **conjunto de test**.

In [ ]:
param_grid = {
    'n_estimators':  [100, 200],
    'learning_rate': [0.01, 0.05, 0.1, 0.5],
    'max_depth':     [1, 2, 3],
    'subsample':     [0.8, 1.0]
}

grid_gb = GridSearchCV(
    GradientBoostingClassifier(random_state=42),
    param_grid,
    cv=5,
    scoring='accuracy',
    n_jobs=-1
)

grid_gb.fit(X_train, y_train)

print(f"Mejores parámetros: {grid_gb.best_params_}")
print(f"Mejor CV score:     {grid_gb.best_score_:.4f}")
print(f"Test score:         {grid_gb.best_estimator_.score(X_test, y_test):.4f}")

### Análisis de Resultados: Optimización de Gradient Boosting

La búsqueda automática ha determinado que la configuración más eficaz para este modelo coincide con los valores que el algoritmo utiliza por defecto.

El uso de una velocidad de aprendizaje de **0.1** junto con **100 árboles** de profundidad **3** permite alcanzar una precisión en validación cruzada del **0.8260**. Sin embargo al aplicar esta misma configuración sobre los datos de prueba el resultado desciende hasta el **0.7877**. El hecho de que el parámetro **subsample** se haya quedado en **1.0** indica que el modelo prefiere utilizar el total de los datos para construir cada árbol en lugar de una muestra reducida.

Este resultado confirma que el Gradient Boosting es muy potente para encontrar patrones en los datos de entrenamiento pero mantiene una brecha de rendimiento cuando se enfrenta a datos que no conoce. A pesar de ser un algoritmo avanzado no ha logrado superar en esta prueba final a los resultados obtenidos por los bosques aleatorios.

**Conclusión**:
El proceso de optimización demuestra que para este conjunto de datos aumentar la complejidad o cambiar la velocidad de aprendizaje no ofrece una mejora sustancial. El modelo es sólido pero parece tener más dificultades para generalizar que otras técnicas de ensamble probadas anteriormente.

---
### 4.5 AdaBoost

En este apartado analizamos el funcionamiento del algoritmo **AdaBoost**. Este modelo también construye árboles de forma secuencial pero utiliza una estrategia basada en pesos. Cuando un árbol comete un error al clasificar a un pasajero el modelo asigna más importancia a ese caso concreto para que el siguiente árbol se esfuerce más en resolverlo correctamente.

Para esta prueba configuramos el modelo con árboles muy simples de profundidad 1 y un total de **100 estimadores**. Calculamos la precisión media con **validación cruzada** y medimos el acierto sobre el **conjunto de test**. Así podemos verificar si centrar el entrenamiento en los ejemplos más difíciles ayuda a conseguir una mejor clasificación global.

In [ ]:
from sklearn.ensemble import AdaBoostClassifier

# AdaBoost con árbol de decisión de profundidad 1 (stump) como estimador base
ada = AdaBoostClassifier(
    estimator=DecisionTreeClassifier(max_depth=1),
    n_estimators=100,
    learning_rate=1.0,
    random_state=42
)

ada.fit(X_train, y_train)
cv_ada = cross_val_score(ada, X_train, y_train, cv=5)

print(f"AdaBoost: CV={cv_ada.mean():.4f} (+/-{cv_ada.std():.4f}), "
      f"Test={ada.score(X_test, y_test):.4f}")

Tabla explorando n_estimators y learning_rate de AdaBoost.

In [ ]:
ada_configs = [(50, 1.0), (100, 1.0), (100, 0.5), (200, 0.1)]
rows = []
for n_est, lr in ada_configs:
    ada = AdaBoostClassifier(
        estimator=DecisionTreeClassifier(max_depth=1),
        n_estimators=n_est,
        learning_rate=lr,
        random_state=42
    )
    ada.fit(X_train, y_train)
    cv = cross_val_score(ada, X_train, y_train, cv=5)
    rows.append({
        'n_estimators': n_est,
        'learning_rate': lr,
        'CV Accuracy': f"{cv.mean():.4f}",
        'Test Accuracy': f"{ada.score(X_test, y_test):.4f}"
    })
pd.DataFrame(rows)

### Análisis de Resultados: AdaBoost

Los datos obtenidos permiten observar cómo influye la velocidad de aprendizaje y la cantidad de árboles en la precisión final del modelo.

* **Efecto de la cantidad de árboles**:
Al subir de 50 a 100 árboles con una velocidad de 1.0 la precisión en el test baja de **0.7933** a **0.7765**. Esto demuestra que añadir más estimadores sin reducir la velocidad de aprendizaje puede perjudicar el rendimiento del modelo en datos nuevos porque este se ajusta demasiado a los ejemplos conocidos.

* **Mejora con velocidad reducida**:
La mejor puntuación en la validación cruzada aparece con **200 árboles** y una velocidad de **0.1** logrando un **0.8091**. Esta combinación es la más estable y consigue recuperar el acierto en el test hasta el **0.7933**. Reducir la velocidad permite que el algoritmo integre la información de forma más gradual y segura.

* **Rendimiento comparativo**:
AdaBoost muestra resultados muy sólidos y constantes que rondan siempre el **80%** de acierto en validación. Aunque es un modelo muy fiable no llega a superar las cifras alcanzadas por el Gradient Boosting o el Random Forest en los apartados anteriores de la práctica.

**Conclusión**:
Para este algoritmo la mejor estrategia consiste en utilizar un número alto de árboles junto con una velocidad de aprendizaje baja. De esta forma el modelo consigue un equilibrio entre el aprendizaje de los casos difíciles y la capacidad de acertar con pasajeros que no ha visto anteriormente.

---
### 4.6 Evaluación Final (Parte 4)

En este apartado final del bloque analizamos el rendimiento del mejor modelo de **Gradient Boosting**. Utilizamos la **matriz de confusión** y el **informe de clasificación** para ver cuántos pasajeros ha clasificado correctamente y en cuáles ha cometido fallos sobre el conjunto de test.

También generamos una tabla comparativa que reúne los mejores modelos vistos hasta ahora. Incluimos el mejor **Random Forest**, el **AdaBoost** y las dos versiones de **Gradient Boosting**. De esta forma podemos comparar la precisión, el F1-Score y el tiempo de ejecución de cada uno para decidir qué algoritmo de ensamble es el más eficaz para este problema.

In [ ]:
best_gb_model = grid_gb.best_estimator_
best_gb_model.fit(X_train, y_train)
y_pred_gb = best_gb_model.predict(X_test)

print(classification_report(y_test, y_pred_gb))

cm = confusion_matrix(y_test, y_pred_gb)
ConfusionMatrixDisplay(confusion_matrix=cm).plot()
plt.title('Matriz de Confusión - Mejor Gradient Boosting')
plt.savefig('images/confusion_matrix_gb.png', dpi=300, bbox_inches='tight')
plt.show()

Tabla comparativa final de la Parte 4.

In [ ]:
# RF mejor config (de Parte 3)
rf_best = grid_rf.best_estimator_
t0 = time()
rf_best.fit(X_train, y_train)
t_rf = time() - t0
cv_rf = cross_val_score(rf_best, X_train, y_train, cv=5)
yp_rf = rf_best.predict(X_test)

# GB por defecto
gb_def = GradientBoostingClassifier(random_state=42)
t0 = time()
gb_def.fit(X_train, y_train)
t_gb_def = time() - t0
cv_gb_def = cross_val_score(gb_def, X_train, y_train, cv=5)
yp_gb_def = gb_def.predict(X_test)

# GB mejor config
gb_best = grid_gb.best_estimator_
t0 = time()
gb_best.fit(X_train, y_train)
t_gb_best = time() - t0
cv_gb_best = cross_val_score(gb_best, X_train, y_train, cv=5)
yp_gb_best = gb_best.predict(X_test)

# AdaBoost (mejor de la tabla)
ada_best = AdaBoostClassifier(
    estimator=DecisionTreeClassifier(max_depth=1),
    n_estimators=100, learning_rate=1.0, random_state=42)
t0 = time()
ada_best.fit(X_train, y_train)
t_ada = time() - t0
cv_ada_best = cross_val_score(ada_best, X_train, y_train, cv=5)
yp_ada = ada_best.predict(X_test)

rows = []
for name, cv, yp, t in [
        ('Random Forest (mejor config)', cv_rf, yp_rf, t_rf),
        ('Gradient Boosting por defecto', cv_gb_def, yp_gb_def, t_gb_def),
        ('Gradient Boosting mejor config', cv_gb_best, yp_gb_best, t_gb_best),
        ('AdaBoost', cv_ada_best, yp_ada, t_ada)]:
    rows.append({
        'Modelo': name,
        'CV Accuracy': f"{cv.mean():.4f}",
        'Test Accuracy': f"{accuracy_score(y_test, yp):.4f}",
        'F1-Score': f"{f1_score(y_test, yp):.4f}",
        'Tiempo (s)': f"{t:.3f}"
    })
pd.DataFrame(rows)

### Análisis Detallado de los Resultados Finales

El análisis conjunto de la matriz de confusión, el informe de clasificación y la tabla comparativa nos permite extraer las siguientes conclusiones técnicas.

#### 1. Rendimiento por Clase (Mejor Gradient Boosting)
Al observar el **Classification Report**, vemos que el modelo no se comporta igual con todos los pasajeros:
* **Clase 0 (No supervivientes):** El modelo es muy eficaz aquí con un **recall del 0.90**. Esto significa que es capaz de detectar a 9 de cada 10 pasajeros que no sobrevivieron.
* **Clase 1 (Supervivientes):** El rendimiento baja considerablemente con un **recall del 0.61**. El modelo tiene dificultades en esta categoría y clasifica por error a 27 supervivientes como si no lo fueran.
* **Equilibrio:** La precisión es idéntica en ambas clases (**0.79**). Esto indica que, cuando el modelo se atreve a predecir una categoría, su nivel de acierto es constante.

#### 2. Comparativa Global de Modelos
La tabla final nos permite comparar el **Gradient Boosting** frente al resto de estrategias de ensamble:
* **Ganador en Precisión:** El **Random Forest (mejor config)** sigue siendo el modelo más sólido con un **Test Accuracy de 0.8045** y la mejor validación cruzada (**0.8287**). Supera al Gradient Boosting en todas las métricas principales.
* **Gradient Boosting (Defecto vs. Optimizado):** Ambos modelos obtienen exactamente el mismo resultado (**0.7877** en test). Esto confirma que los parámetros de serie ya eran los más adecuados para este problema y que el proceso de búsqueda no ha encontrado una combinación superior.
* **AdaBoost:** Es el modelo con menor precisión en el test (**0.7765**). Aunque su **F1-Score (0.7183)** es mejor que el del Gradient Boosting, su capacidad de acierto global es más limitada.

#### 3. Eficiencia y Tiempo
En cuanto al tiempo de ejecución, los modelos de **Boosting** son notablemente más rápidos que el Random Forest optimizado. El Gradient Boosting tarda solo **0.135 segundos** frente a los **0.359 segundos** del bosque aleatorio. No obstante, al tratarse de un dataset pequeño, esta diferencia de tiempo es insignificante y no justifica la pérdida de precisión.

**Conclusión Final:**
Tras probar todas las técnicas de aumento de gradiente y pesos, el **Random Forest** del bloque anterior se mantiene como la solución más equilibrada y fiable para predecir la supervivencia en el Titanic.

---
## Parte Final: Comparación y Evaluación Global

### Comparativa Final de Modelos

En este apartado final reunimos las mejores configuraciones de cada uno de los modelos analizados a lo largo de la práctica. El objetivo es realizar una competición directa entre el **SVM**, el **Árbol de Decisión**, el **Random Forest**, el **Gradient Boosting** y el **AdaBoost** para determinar cuál es el sistema más eficaz para predecir la supervivencia en el Titanic.

Evaluamos cada algoritmo utilizando métricas de precisión, exhaustividad (**recall**) y el valor **F1-Score**. También medimos el tiempo que tarda cada modelo en entrenarse para comprobar su eficiencia. Al terminar el proceso identificamos automáticamente el modelo con mejor rendimiento en el conjunto de test y generamos su matriz de confusión definitiva para analizar sus aciertos y errores en detalle.

In [ ]:
best_models = {
    'SVM (mejor config)':      grid_rbf.best_estimator_,
    'Decision Tree':           grid_tree.best_estimator_,
    'Random Forest':           grid_rf.best_estimator_,
    'Gradient Boosting':       grid_gb.best_estimator_,
    'AdaBoost':                AdaBoostClassifier(
        estimator=DecisionTreeClassifier(max_depth=1),
        n_estimators=100, learning_rate=1.0, random_state=42)
}

rows = []
for name, model in best_models.items():
    t0 = time()
    model.fit(X_train, y_train)
    t1 = time()
    cv  = cross_val_score(model, X_train, y_train, cv=5, scoring='accuracy')
    yp = model.predict(X_test)
    rows.append({
        'Modelo': name,
        'CV Accuracy': f"{cv.mean():.4f} (+/-{cv.std():.4f})",
        'Test Accuracy': f"{accuracy_score(y_test, yp):.4f}",
        'Precision': f"{precision_score(y_test, yp):.4f}",
        'Recall': f"{recall_score(y_test, yp):.4f}",
        'F1-Score': f"{f1_score(y_test, yp):.4f}",
        'Tiempo (s)': f"{t1-t0:.3f}"
    })
pd.DataFrame(rows)

Matriz de confusión del mejor modelo global.

In [ ]:
# Encontrar el mejor modelo global por Test Accuracy
best_name = None
best_acc = 0
best_model_global = None
for name, model in best_models.items():
    model.fit(X_train, y_train)
    acc = model.score(X_test, y_test)
    if acc > best_acc:
        best_acc = acc
        best_name = name
        best_model_global = model

print(f"Mejor modelo global: {best_name} (Test Accuracy: {best_acc:.4f})")

y_pred_final = best_model_global.predict(X_test)
print(classification_report(y_test, y_pred_final))

cm = confusion_matrix(y_test, y_pred_final)
ConfusionMatrixDisplay(confusion_matrix=cm).plot()
plt.title(f'Matriz de Confusión - {best_name}')
plt.savefig('images/confusion_matrix_final.png', dpi=300, bbox_inches='tight')
plt.show()

### Conclusiones y Selección del Modelo Final

Tras completar el análisis de todos los algoritmos, la tabla comparativa permite identificar al modelo con mejor desempeño para este conjunto de datos.

#### Análisis de la Tabla Comparativa
Al observar las métricas de todos los modelos optimizados, extraemos las siguientes conclusiones:

* **El modelo con mejor desempeño:** El **SVM (mejor config)** se posiciona como el mejor modelo global. Logra la **validación cruzada más alta (0.8385)** y una precisión en el **test de 0.8045**. Aunque el **Random Forest** alcanza la misma precisión en el test, el SVM es un modelo más sencillo y mucho más rápido de entrenar.
* **Precisión y F1-Score:** El **Random Forest** obtiene el mejor **F1-Score (0.7244)**, lo que indica un equilibrio superior entre precisión y recall. Sin embargo, el SVM destaca por tener la **Precision más alta (0.8400)** al predecir supervivientes, cometiendo muy pocos errores de falsos positivos.
* **Modelos de Boosting:** Tanto el **Gradient Boosting** como el **AdaBoost** se quedan ligeramente por detrás en precisión final. A pesar de ser algoritmos más complejos, no han conseguido superar la barrera del 80% de acierto en este dataset específico.

#### Evaluación del Modelo Ganador: SVM
La matriz de confusión del **SVM** confirma su solidez:
1. **Clase 0 (No supervivientes):** Acierta con 102 pasajeros y solo falla en 8 casos. Es un modelo excelente para identificar quiénes no sobrevivieron.
2. **Clase 1 (Supervivientes):** Identifica correctamente a 42 pasajeros. Aunque deja de detectar a 27 (recall de 0.61), cuando el modelo predice que alguien sobrevive, acierta el **84%** de las veces.

**Conclusión Final**:
Para este problema del Titanic, el **SVM con núcleo RBF** es la opción más recomendable. Ofrece el mejor equilibrio entre capacidad de generalización (CV Accuracy), acierto real sobre datos nuevos (Test Accuracy) y eficiencia técnica, ya que su tiempo de ejecución es prácticamente instantáneo (0.015 segundos).